In [1]:
from pathlib import Path
import csv, io, json, math, re, collections, random, warnings
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle, Polygon
from scipy.ndimage import gaussian_filter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader, simpleSplit
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import fitz

c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [2]:
DEMO_MODE = False

TRACKMAN_FILE = Path(r"C:\Users\brend\OneDrive\School OneDrive\2025 CCBL Trackman Data.xlsx")
TRACKMAN_SHEET = 0
OUTPUT_DIR = Path('FAL_Hitter_Advance_Hitters')
TEAM_FILTER = None
DATE_START = '2025-01-01'
DATE_END = '2026-12-31'
ZONE_BOTTOM, ZONE_TOP, ZONE_HALF_WIDTH = 1.5, 3.5, 0.83
HARD_HIT_MPH = 95.0
MIN_SPRAY_DISTANCE_FT = 5.0
DIRECTION_SIGN = 1
MIN_DENSITY_N = 12
KPI_FIELDS = ['PITCHES','BIP','SWING%','WHIFF%','CHASE%','Z-SWING%','Z-CONTACT','AVG EV','MAX EV','HH%']

HITTERS = [
    dict(name='Charlie Bates', batter_id='807287', bats='L', team='HYA_HAR', season={}, notes={}, official_splits={}),
    dict(name='Levi Clark', batter_id='815155', bats='R', team='WAR_GAT', season={}, notes={}, official_splits={}),
    dict(name='Jake Hanley', batter_id='827368', bats='L', team='CHA_ANG / COT_KET', season={}, notes={}, official_splits={}),
]
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Mode:', 'SYNTHETIC DEMONSTRATION' if DEMO_MODE else 'REAL CCBL 2025–2026 TRACKMAN DATA')
print('Output folder:', OUTPUT_DIR.resolve())

Mode: REAL CCBL 2025–2026 TRACKMAN DATA
Output folder: C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters


In [3]:
REQUIRED_COLUMNS = ['Batter', 'PitchCall']
ALIASES = {
 'Batter':['Batter','BatterName','batter_name','Player','playerFullName'],
 'BatterId':['BatterId','BatterID','Batter_ID'],
 'BatterTeam':['BatterTeam','BattingTeam','BatterTeamCode'],
 'PitcherThrows':['PitcherThrows','PitcherThrow','PitcherHand','PitcherSide','Throws'],
 'BatterSide':['BatterSide','BatterHand','Bats'],
 'TaggedPitchType':['TaggedPitchType','PitchType','AutoPitchType','PitchTag'],
 'PitchCall':['PitchCall','PitchResult','PitchOutcome'],
 'PlayResult':['PlayResult','Result','PlateAppearanceResult'],
 'Balls':['Balls','BallCount','PrePitchBalls'],
 'Strikes':['Strikes','StrikeCount','PrePitchStrikes'],
 'PlateLocSide':['PlateLocSide','PlateSide','PlateX'],
 'PlateLocHeight':['PlateLocHeight','PlateHeight','PlateZ'],
 'ExitSpeed':['ExitSpeed','ExitVelocity','ExitVelo','EV'],
 'Angle':['Angle','LaunchAngle','LA'],
 'Direction':['Direction','SprayAngle','Spray'],
 'Distance':['Distance','HitDistance','CarryDistance'],
 'TaggedHitType':['TaggedHitType','HitType','BattedBallType'],
 'Date':['Date','GameDate','PitchDate'],
}
FAMILY_MAP = {
 'fastball':'FB','four-seam':'FB','fourseam':'FB','four-seamfastball':'FB',
 'fourseamfastball':'FB','4-seam':'FB','sinker':'FB','twoseamfastball':'FB',
 'two-seam':'FB','2-seam':'FB','cutter':'FB',
 'slider':'BB','sweeper':'BB','curveball':'BB','knucklecurve':'BB','slurve':'BB',
 'changeup':'OS','splitter':'OS','forkball':'OS','knuckleball':'OS'
}
NUMERIC=['Balls','Strikes','PlateLocSide','PlateLocHeight','ExitSpeed','Angle','Direction','Distance']
CANONICAL=list(ALIASES)

def normalize_name(value):
    s=str(value or '').strip()
    if ',' in s:
        a,b=s.split(',',1)
        s=b.strip()+' '+a.strip()
    return re.sub(r'[^a-z0-9]', '', s.casefold())

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_-]+','_',str(value)).strip('_') or 'Hitter'

def decode_side(value):
    s=str(value or '').strip().casefold()
    if s in ('r','right','rhp','rh'): return 'RHP'
    if s in ('l','left','lhp','lh'): return 'LHP'
    return None

def pitch_family(value):
    s=re.sub(r'[ _]','',str(value or '').strip().casefold())
    s=s.replace('fourseamfastball','fourseamfastball')
    mapping={re.sub(r'[ _]','',key):v for key,v in FAMILY_MAP.items()}
    return mapping.get(s,'OTHER')

def rename_fields(df):
    df=df.loc[:,~pd.Index(df.columns).duplicated(keep='first')].copy()
    find={}
    for c in df.columns:
        find.setdefault(re.sub(r'[^a-z0-9]','',str(c).lower()), c)
    ren={}
    for desired,candidates in ALIASES.items():
        for c in candidates:
            old=find.get(re.sub(r'[^a-z0-9]','',c.lower()))
            if old is not None:
                ren[old]=desired; break
    return df.rename(columns=ren)

def parse_trackman_date(values):
    values = values.astype(object)

    is_excel_serial = values.map(lambda v: isinstance(v, (int,float,np.integer,np.floating)) or (isinstance(v,str) and bool(re.fullmatch(r'\d+(?:\.\d+)?',v.strip()))))
    numerics = pd.to_numeric(values.where(is_excel_serial), errors='coerce')
    result = pd.to_datetime(values.where(numerics.isna()), errors='coerce')
    if numerics.notna().any():
        result.loc[numerics.notna()] = pd.to_datetime(numerics.loc[numerics.notna()], unit='D', origin='1899-12-30', errors='coerce')
    return result

def stream_trackman(path, profiles, team=None, sheet=0, date_start=None, date_end=None):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Missing TrackMan file: {path}')
    want={normalize_name(p['name']) for p in profiles}
    want_ids={str(p['batter_id']) for p in profiles if p.get('batter_id')}
    stats={'source':str(path),'rows_scanned':0,'matched_rows':0, 'skipped_team':0, 'skipped_date':0}
    matches=[]
    def accept(df):
        nonlocal matches
        df=rename_fields(df)
        missing=[v for v in REQUIRED_COLUMNS if v not in df.columns]
        if missing: raise ValueError('Missing required TrackMan columns: '+', '.join(missing))
        stats['rows_scanned']+=len(df)
        df=df[(df['Batter'].astype(str).map(normalize_name).isin(want)) | (df['BatterId'].astype(str).isin(want_ids) if 'BatterId' in df.columns else False)].copy()
        if team and 'BatterTeam' in df:
            old=len(df)
            df=df[df['BatterTeam'].astype(str).str.strip().str.casefold()==str(team).strip().casefold()]
            stats['skipped_team']+=old-len(df)
        if (date_start or date_end) and 'Date' in df:
            old = len(df)
            day = pd.to_datetime(parse_trackman_date(df['Date']), errors='coerce')

            mask = day.notna()

            if date_start:
                mask &= day >= pd.Timestamp(date_start)

            if date_end:
                mask &= day < pd.Timestamp(date_end) + pd.Timedelta(days=1)

            df = df.loc[mask].copy()
            stats['skipped_date'] += old - len(df)
        if not df.empty:
            stats['matched_rows']+=len(df)
            matches.append(df.reindex(columns=CANONICAL))
    ext=path.suffix.casefold()
    if ext in ('.csv','.txt'):
        for chunk in pd.read_csv(path,chunksize=50000,low_memory=False,encoding='utf-8-sig'):
            accept(chunk)
    elif ext in ('.xlsx','.xlsm'):
        from openpyxl import load_workbook
        workbook=load_workbook(path,read_only=True,data_only=True)
        ws=workbook.worksheets[sheet] if isinstance(sheet,int) else workbook[sheet]
        rows=ws.iter_rows(values_only=True)
        headers=next(rows,None)
        if headers is None: raise ValueError('Empty Excel sheet.')
        chunk=[]
        for row in rows:
            chunk.append(row)
            if len(chunk)>=20000:
                accept(pd.DataFrame(chunk,columns=headers));chunk=[]
        if chunk: accept(pd.DataFrame(chunk,columns=headers))
        workbook.close()
    else:
        raise ValueError('Supported TrackMan files: .csv, .xlsx, .xlsm')
    data=pd.concat(matches,ignore_index=True) if matches else pd.DataFrame(columns=CANONICAL)
    for col in NUMERIC:
        data[col]=pd.to_numeric(data[col],errors='coerce')
    data['Hand']=data['PitcherThrows'].map(decode_side)
    data['Family']=data['TaggedPitchType'].map(pitch_family)
    data['Call']=data['PitchCall'].fillna('').astype(str).str.lower().str.replace(r'[^a-z]','',regex=True)
    stats['missing_pitch_calls']=int(data['Call'].eq('').sum())
    stats['missing_pitcher_hand']=int(data['Hand'].isna().sum())
    stats['pitch_type_unclassified']=int(data['Family'].eq('OTHER').sum())

    data['Swing']=data['Call'].str.startswith(('strikeswinging','foulball','inplay')) | data['Call'].isin(['swingingstrike','foul','foultip'])
    data['Whiff']=data['Call'].str.startswith('strikeswinging') | data['Call'].isin(['swingingstrike','swingingstrikeblocked'])
    data['InPlay']=data['Call'].str.startswith('inplay')
    x=data['PlateLocSide'].abs()
    y=data['PlateLocHeight']
    data['ZoneKnown']=x.notna() & y.notna()
    data['InZone']=(x<=ZONE_HALF_WIDTH) & (y>=ZONE_BOTTOM) & (y<=ZONE_TOP) & data['ZoneKnown']
    data['HardHit']=data['InPlay'] & data['ExitSpeed'].ge(HARD_HIT_MPH)
    data['Count']=data['Balls'].fillna(-1).astype(int).astype(str)+'-'+data['Strikes'].fillna(-1).astype(int).astype(str)
    return data,stats

def generate_synthetic_demo(path, name='Example Hitter', n=1050, seed=42):
    rng=np.random.default_rng(seed)
    calls=['BallCalled','StrikeCalled','StrikeSwinging','FoulBall','InPlay']
    probs=[.335,.170,.180,.158,.157]
    rows=[]
    for i in range(n):
        call=rng.choice(calls,p=probs)
        hand=rng.choice(['Right','Left'],p=[.67,.33])
        ptype=rng.choice(['Fastball','Slider','Changeup','Sinker'],p=[.40,.26,.19,.15])
        balls=int(rng.integers(0,4)); strikes=int(rng.integers(0,3))
        loc_x=float(rng.normal(0.12,0.82))
        loc_y=float(rng.normal(2.52,0.80))
        if call=='InPlay':
            ev=float(np.clip(rng.normal(88.5 if ptype in ('Fastball','Sinker') else 85,12),48,115))
            la=float(np.clip(rng.normal(13,24),-45,78))
            direction=float(np.clip(rng.normal(9,24),-53,53))
            distance=float(np.clip(ev*2.5+la*.7+rng.normal(0,45),75,390))
            btype='GroundBall' if la<10 else ('LineDrive' if la<25 else ('FlyBall' if la<50 else 'Popup'))
        else:ev=la=direction=distance=None;btype=''
        rows.append(dict(Date='2025-06-15',Batter=name,BatterTeam='DEMO',BatterSide='Left',
                         PitcherThrows=hand,TaggedPitchType=ptype,PitchCall=call,
                         Balls=balls,Strikes=strikes,PlateLocSide=loc_x,PlateLocHeight=loc_y,
                         ExitSpeed=ev,Angle=la,Direction=direction,Distance=distance,
                         TaggedHitType=btype))
    pd.DataFrame(rows).to_csv(path,index=False)
    return path

if DEMO_MODE:
    TRACKMAN_FILE=generate_synthetic_demo(OUTPUT_DIR/'demo_synthetic_trackman.csv',HITTERS[0]['name'])

DATA, INPUT_AUDIT = stream_trackman(TRACKMAN_FILE,HITTERS,TEAM_FILTER,TRACKMAN_SHEET,DATE_START,DATE_END)
print('TrackMan ingestion:',json.dumps(INPUT_AUDIT,indent=2))
if DATA.empty: raise ValueError('No matching hitter pitches. Check HITTERS names, TEAM_FILTER, date filters, and input path.')

if 'BatterTeam' in DATA and DATA['BatterTeam'].astype(str).str.strip().str.upper().eq('FAL_COM').any():
    raise ValueError('A selected hitter appeared for FAL_COM in the supplied source; reject the entire hitter rather than only FAL_COM rows.')

TrackMan ingestion: {
  "source": "C:\\Users\\brend\\OneDrive\\School OneDrive\\2025 CCBL Trackman Data.xlsx",
  "rows_scanned": 114410,
  "matched_rows": 2643,
  "skipped_team": 0,
  "skipped_date": 10,
  "missing_pitch_calls": 0,
  "missing_pitcher_hand": 1,
  "pitch_type_unclassified": 11
}


In [4]:
def pct(n,d): return (100*float(n)/float(d)) if d else None

def average(values):
    vv=pd.to_numeric(values,errors='coerce').dropna()
    return float(vv.mean()) if len(vv) else None

def ratio(num,den):
    return pct(int(num),int(den))

def basic_rates(df):
    known=df[df['Call'].ne('')]
    loc=known[known['ZoneKnown']]
    zon=loc[loc['InZone']];out=loc[~loc['InZone']]
    swings=int(known['Swing'].sum())
    return {
        'N':len(df), 'swing':ratio(swings,len(known)),
        'z_swing':ratio(zon['Swing'].sum(),len(zon)),
        'whiff':ratio(df['Whiff'].sum(),swings),
        'chase':ratio(out['Swing'].sum(),len(out)),
        'z_contact':ratio(int(zon['Swing'].sum())-int(zon['Whiff'].sum()),int(zon['Swing'].sum())),
        'known_zone_n':len(loc),'swing_n':swings,'known_call_n':len(known),
    }

def batted_ball_type(df):
    b=df.copy()
    hit=b.get('TaggedHitType',pd.Series(index=b.index,dtype=object)).fillna('').astype(str).str.lower().str.replace(r'[^a-z]','',regex=True)
    ang=pd.to_numeric(b['Angle'],errors='coerce')
    labels=np.full(len(b),'UNKNOWN',dtype=object)
    for names,label in [(['groundball','grounder'],'GB'),(['linedrive','line'],'LD'),
                        (['flyball','fly'],'FB'),(['popup','popfly','popout'],'PU')]:
        labels[hit.isin(names).to_numpy()]=label
    unknown=(labels=='UNKNOWN') & ang.notna().to_numpy()
    labels[unknown & (ang.to_numpy()<10)]='GB'
    labels[unknown & (ang.to_numpy()>=10) & (ang.to_numpy()<25)]='LD'
    labels[unknown & (ang.to_numpy()>=25) & (ang.to_numpy()<50)]='FB'
    labels[unknown & (ang.to_numpy()>=50)]='PU'
    b['BBType']=labels
    return b

def direction_class(df, bats):
    a=pd.to_numeric(df['Direction'],errors='coerce')*DIRECTION_SIGN
    hand=str(bats or '').upper()[:1]

    if hand=='R': return np.select([a<-15,a>15],['PULL','OPPO'],default='MIDDLE')
    if hand=='L': return np.select([a>15,a<-15],['PULL','OPPO'],default='MIDDLE')
    if 'BatterSide' in df:
        sides=df['BatterSide'].fillna('').astype(str).str.lower().str[0]
        return np.select([(sides=='r') & (a<-15), (sides=='l') & (a>15),
                          (sides=='r') & (a>15), (sides=='l') & (a<-15),
                          sides.isin(['r','l'])],
                         ['PULL','PULL','OPPO','OPPO','MIDDLE'],default='UNKNOWN')
    return np.full(len(df),'UNKNOWN')

def contact_metrics(df,bats='L'):
    bip=batted_ball_type(df[df['InPlay']].copy())
    ev=bip[bip['ExitSpeed'].notna()]
    n=len(ev)
    comp=bip[bip['BBType']!='UNKNOWN']
    d=bip[bip['Direction'].notna()].copy()
    d['DirClass']=direction_class(d,bats)
    d=d[d['DirClass']!='UNKNOWN']
    output={'bip':len(bip),'ev_n':n,'ev':average(ev['ExitSpeed']),
            'max_ev':float(ev['ExitSpeed'].max()) if n else None,
            'hh':ratio(ev['HardHit'].sum(),n)}
    for typ in ['GB','LD','FB','PU']:
        output[typ.lower()]=ratio((comp['BBType']==typ).sum(),len(comp))
    for key in ['PULL','MIDDLE','OPPO']:
        output[key.lower()]=ratio((d['DirClass']==key).sum(),len(d))
    output['bb_type_n']=len(comp); output['direction_n']=len(d)
    return output

def count_rows(df,hand,official=None):
    df=df[df['Hand']==hand]
    groups=[('Overall',df),('0-0',df[(df['Balls']==0)&(df['Strikes']==0)]),
           ('3-2',df[(df['Balls']==3)&(df['Strikes']==2)]),
           ('Behind',df[df['Balls']<df['Strikes']]),
           ('Ahead',df[df['Balls']>df['Strikes']])]
    out=[]
    for label,sub in groups:
        v=basic_rates(sub)
        override=(official or {}).get(label,{})
        v.update(group=label,bip=int(sub['InPlay'].sum()))
        out.append(v)
    return out

def pitch_type_rows(df,bats,max_types=4):
    out=[]
    data=df.copy()
    data['PitchType']=data['TaggedPitchType'].fillna('Unknown').astype(str).str.strip()
    for pitch,n in data['PitchType'].value_counts().items():
        sub=data[data['PitchType']==pitch]
        if not len(sub[sub['InPlay']]):continue
        out.append(dict(pitch=pitch,pitches=int(n),**contact_metrics(sub,bats)))
        if len(out)>=max_types:break
    return out

def family_rows(df,bats):
    out=[]
    for fam,label in [('FB','FASTBALLS'),('BB','BREAKING BALLS'),('OS','OFFSPEED')]:
        sub=df[df['Family']==fam]
        if not len(sub):
            continue
        out.append(dict(family=label,pitches=len(sub),**contact_metrics(sub,bats)))
    return out

In [5]:
from reportlab.lib.colors import HexColor
W,H=14*72,8*72
MAROON='#842638'; WHITE='#FFFFFF'; PAPER='#F4F6F9'; INK='#18191C'; BODY='#273341'; MUTED='#607082'; LINE='#D9E0E7'; LIGHT='#EAF0F5'; BLACK='#131418'
ROSE='#B8475F'; BLUE='#2274A5'; GREEN='#4D8766'
FONT, FONT_BOLD='Helvetica','Helvetica-Bold'
for regular,bold in [('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf','/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'),
                     ('C:/Windows/Fonts/arial.ttf','C:/Windows/Fonts/arialbd.ttf')]:
    if Path(regular).is_file() and Path(bold).is_file():
        try:
            pdfmetrics.registerFont(TTFont('HitterRegular',regular))
            pdfmetrics.registerFont(TTFont('HitterBold',bold))
            FONT,FONT_BOLD='HitterRegular','HitterBold';break
        except Exception:pass

def shape(c,x,y,w,h,color=WHITE,stroke=None,r=0):
    c.setFillColor(HexColor(color));c.setStrokeColor(HexColor(stroke or color))
    if r:c.roundRect(x*72,H-(y+h)*72,w*72,h*72,r*72,stroke=int(bool(stroke)),fill=1)
    else:c.rect(x*72,H-(y+h)*72,w*72,h*72,stroke=int(bool(stroke)),fill=1)

def txt(c,x,y,s,size=9,color=BODY,bold=False,align='left'):
    c.setFont(FONT_BOLD if bold else FONT,size);c.setFillColor(HexColor(color))
    fx,fy=x*72,H-y*72-size
    {'left':c.drawString,'center':c.drawCentredString,'right':c.drawRightString}[align](fx,fy,str(s))

def fit_txt(c,x,y,s,max_w,size=9,min_size=6.5,color=BODY,bold=False,align='left'):
    font=FONT_BOLD if bold else FONT
    while size>min_size and pdfmetrics.stringWidth(str(s),font,size)>max_w*72:size-=0.3
    txt(c,x,y,s,size,color,bold,align)

def wrapped(c,s,x,y,width,size=7.8,leading=11.4,max_lines=None,color=BODY,bold=False):
    font=FONT_BOLD if bold else FONT
    lines=simpleSplit(str(s or ''),font,size,width*72)
    if max_lines is not None:
        if len(lines)>max_lines and max_lines>0:
            lines=lines[:max_lines]

            last=lines[-1]
            while last and pdfmetrics.stringWidth(last+' ...',font,size)>width*72:last=last[:-1]
            lines[-1]=last+' ...'
    for ix,line in enumerate(lines):txt(c,x,y+ix*leading/72,line,size,color,bold)
    return len(lines)*leading/72

def card(c,x,y,w,h,title=None,subtitle=None):
    shape(c,x,y,w,h,WHITE,LINE)
    shape(c,x,y,.04,h,BLACK)
    if title:txt(c,x+.17,y+.12,title,9.0,INK,True)
    if subtitle:fit_txt(c,x+w-.15,y+.15,subtitle,w*.45,6.5,6.2,MUTED,align='right')

def fmt(v,kind='num',digits=1):
    if v is None or pd.isna(v):return '—'
    if kind=='pct':return f'{v:.{digits}f}%'
    if kind=='int':return str(int(v))
    return f'{v:.{digits}f}'

def render_fig(fig):
    b=io.BytesIO();fig.savefig(b,format='png',dpi=170,facecolor=WHITE,bbox_inches=None,pad_inches=0)
    plt.close(fig);b.seek(0)
    return Image.open(b).convert('RGB')

def image_in(c,img,x,y,w,h,align='center'):
    iw,ih=img.size; scale=min(w*72/iw,h*72/ih)
    actualw=iw*scale;actualh=ih*scale
    xp=x*72+(w*72-actualw)/2; yp=H-(y+h)*72+(h*72-actualh)/2
    c.drawImage(ImageReader(img),xp,yp,width=actualw,height=actualh,mask='auto')

def axis_clean(ax):
    ax.grid(color='#E5EAEF',lw=.65,alpha=.75)
    for spine in ax.spines.values():spine.set_visible(False)
    ax.tick_params(length=0,labelsize=7,colors=MUTED)

def plot_spray(df,bats):
    """Field-authentic spray. Plot ONLY observed distances; direction-only case is a schematic."""
    from matplotlib.patches import Circle
    bip=df[df['InPlay'] & df['Direction'].notna()].copy()
    bip=bip[(bip['Distance'].gt(MIN_SPRAY_DISTANCE_FT)) & ((bip['ExitSpeed'].isna()) | bip['ExitSpeed'].gt(60)) & bip['Direction'].abs().le(55)].copy()
    fig,ax=plt.subplots(figsize=(5.5,3.0),dpi=170)
    fig.subplots_adjust(left=.015,right=.985,top=.975,bottom=.02)
    ax.set_facecolor('#F0F3F2')

    arc=np.linspace(-np.pi/4,np.pi/4,241)
    radius=400-75*(np.abs(arc)/(np.pi/4))**1.5
    xwall=radius*np.sin(arc);ywall=radius*np.cos(arc)
    ax.add_patch(Polygon(np.vstack([[[0,0]],np.column_stack([xwall,ywall])]),
                         closed=True,facecolor='#D7E8CE',edgecolor='none',zorder=0))

    for r0,r1,face in [(140,210,'#D3E5C9'),(210,280,'#DDEBD5'),(280,347,'#D3E5C9')]:
        inner=np.column_stack([r0*np.sin(arc),r0*np.cos(arc)])
        outer=np.column_stack([r1*np.sin(arc),r1*np.cos(arc)])
        ax.add_patch(Polygon(np.vstack([inner,outer[::-1]]),facecolor=face,edgecolor='none',zorder=.5))
    track_inner=np.column_stack([(radius-10)*np.sin(arc),(radius-10)*np.cos(arc)])
    wall_outer=np.column_stack([xwall,ywall])
    ax.add_patch(Polygon(np.vstack([track_inner,wall_outer[::-1]]),facecolor='#C7AC8B',edgecolor='none',zorder=1.1))
    ax.plot(xwall,ywall,color='#4F7661',lw=2.4,zorder=2)

    b=90/np.sqrt(2)
    home=(0,0);first=(b,b);second=(0,2*b);third=(-b,b)
    ax.add_patch(Polygon([home,first,second,third],closed=True,facecolor='#C99C74',edgecolor='#B78662',lw=1.2,zorder=2))
    inner=np.array([home,first,second,third])*.68+np.array([0,64])*.32
    ax.add_patch(Polygon(inner,closed=True,facecolor='#C6DDBF',edgecolor='none',zorder=2.2))

    for a,bp in [(home,first),(first,second),(second,third),(third,home)]:
        ax.plot([a[0],bp[0]],[a[1],bp[1]],color='#F9F6EF',lw=1.5,zorder=3)
    ax.add_patch(Circle((0,59.0),9.0,facecolor='#D7AF88',edgecolor='#B88F6B',lw=.6,zorder=3))

    ax.add_patch(Rectangle((-2.25,60.5),4.5,1.1,facecolor='#F7F2E8',edgecolor='#B6AB96',lw=.4,zorder=3.5))
    for bx,by in (first,second,third):
        ax.add_patch(Rectangle((bx-4,by-4),8,8,angle=45,facecolor=WHITE,edgecolor='#A2A5A0',lw=.5,zorder=4))
    ax.add_patch(Polygon([(-5,0),(5,0),(5,-5),(0,-9),(-5,-5)],closed=True,facecolor=WHITE,edgecolor='#A8AAA6',lw=.65,zorder=4))
    for a in (-np.pi/4,np.pi/4):
        ax.plot([0,326*np.sin(a)],[0,326*np.cos(a)],color='#F7F9F1',lw=1.45,zorder=2.5)

    direction=bip['Direction'].to_numpy(dtype=float)*DIRECTION_SIGN
    dist=pd.to_numeric(bip['Distance'],errors='coerce').to_numpy(dtype=float)
    valid=np.isfinite(dist)&(dist>MIN_SPRAY_DISTANCE_FT)&(dist<=470)&(np.abs(direction)<=55)
    if valid.sum():
        theta=np.deg2rad(direction[valid]);rr=dist[valid]
        xx=rr*np.sin(theta); yy=rr*np.cos(theta)
        hard=bip['HardHit'].to_numpy(dtype=bool)[valid]
        ax.scatter(xx[~hard],yy[~hard],s=23,c='#376C92',alpha=.69,edgecolors=WHITE,linewidths=.34,zorder=6)
        ax.scatter(xx[hard],yy[hard],s=25,c=MAROON,alpha=.84,edgecolors=WHITE,linewidths=.38,zorder=7)
    elif len(bip):

        bins=np.linspace(-45,45,13)
        counts,_=np.histogram(direction,bins)
        peak=max(1,counts.max())
        for n,lo,hi in zip(counts,bins[:-1],bins[1:]):
            if n==0:continue
            angle=np.deg2rad(np.linspace(lo,hi,9))
            rr=105+(n/peak)*155
            sector=np.vstack([[[0,0]],np.column_stack([rr*np.sin(angle),rr*np.cos(angle)])])
            ax.add_patch(Polygon(sector,closed=True,facecolor=MAROON,alpha=.16+.34*n/peak,edgecolor=WHITE,lw=.5,zorder=5))
        ax.text(.985,.025,'DIRECTION ONLY · NOT DISTANCE',fontsize=7,color=MAROON,ha='right',transform=ax.transAxes)
    else:
        ax.text(0,295,'NO SPRAY DATA',fontsize=9,color=MUTED,ha='center',zorder=5)
    for label,x,y in [('LF',-177,328),('CF',0,391),('RF',177,328)]:
        ax.text(x,y,label,fontsize=8,fontweight='bold',color='#425F50',ha='center',zorder=8)
    ax.set_xlim(-310,310);ax.set_ylim(-29,428)
    ax.set_aspect('equal');ax.axis('off')
    return render_fig(fig)

def plot_ev_la(df):
    b=df[df['InPlay'] & df['Angle'].notna() & df['ExitSpeed'].notna()]
    fig,ax=plt.subplots(figsize=(4.7,3.10),dpi=170)
    fig.subplots_adjust(left=.13,right=.97,bottom=.17,top=.93)
    ax.axhspan(HARD_HIT_MPH,125,color='#F9ECEF',alpha=.88,zorder=0)
    if not len(b):ax.text(.50,.50,'NO EV / LA DATA',transform=ax.transAxes,ha='center',va='center',fontsize=11,color=MUTED)
    if len(b):
        non=b[~b['HardHit']];hard=b[b['HardHit']]
        ax.scatter(non['Angle'],non['ExitSpeed'],s=20,c='#89A3B4',alpha=.66,edgecolors='none')
        ax.scatter(hard['Angle'],hard['ExitSpeed'],s=23,c=MAROON,alpha=.75,edgecolors='white',linewidths=.24)
    ax.axhline(HARD_HIT_MPH,color=MAROON,lw=.9,ls='--')
    for a in [10,25,50]:ax.axvline(a,color='#B7C2CC',lw=.7,ls='--')
    ax.set_xlim(-45,80);ax.set_ylim(45,120)
    ax.set_xlabel('Launch angle (°)',fontsize=8,color=MUTED)
    ax.set_ylabel('Exit velocity (mph)',fontsize=8,color=MUTED)
    axis_clean(ax)
    ax.text(.98,.95,f'HH ≥ {HARD_HIT_MPH:g} mph',transform=ax.transAxes,ha='right',va='top',color=MAROON,fontsize=8)
    return render_fig(fig)

def draw_bar(c,x,y,w,label,value,color=MAROON):
    txt(c,x,y,label,7.5,MUTED,True)
    txt(c,x+w,y,fmt(value,'pct',0),7.6,INK,True,'right')
    shape(c,x,y+.24,w,.095,LIGHT)
    if value is not None and pd.notna(value):shape(c,x,y+.24,w*min(1,max(0,value/100)),.095,color)

def heatmap_fig(sub,is_whiff):
    sub=sub[sub['PlateLocSide'].notna() & sub['PlateLocHeight'].notna()]
    fig,ax=plt.subplots(figsize=(3.30,1.90),dpi=160)
    fig.subplots_adjust(left=.06,right=.97,bottom=.04,top=.98)
    ax.set_facecolor('#F5F8FA')
    xx=sub['PlateLocSide'].to_numpy(dtype=float)
    yy=sub['PlateLocHeight'].to_numpy(dtype=float)
    n=len(xx)
    palette=LinearSegmentedColormap.from_list('Falmouth', ['#EEF3F7','#CADFE9','#7EAEC4',BLUE]) if is_whiff else LinearSegmentedColormap.from_list('Damage',['#F6EDEF','#E8BBC4','#BE6579',MAROON])
    if n>=MIN_DENSITY_N:
        z,xedge,yedge=np.histogram2d(xx,yy,bins=(66,86),range=[[-2.0,2.0],[.45,4.65]])
        z=gaussian_filter(z,sigma=(4.4,5.0))
        z=z/max(z.max(),1e-9)
        alpha=np.clip((z-.025)/.12,0,.94)
        ax.imshow(z.T,origin='lower',extent=[-2.,2.,.45,4.65],cmap=palette,
                  alpha=alpha.T,interpolation='bilinear',aspect='auto',vmin=0,vmax=.75)
    elif n>0:
        ax.scatter(xx,yy,c=BLUE if is_whiff else MAROON,s=15,alpha=.70,edgecolors=WHITE,linewidths=.3)
    ax.add_patch(Rectangle((-ZONE_HALF_WIDTH,ZONE_BOTTOM),2*ZONE_HALF_WIDTH,ZONE_TOP-ZONE_BOTTOM,
                           facecolor='none',edgecolor=BLACK,lw=1.2))
    ax.set_xlim(-2.0,2.0);ax.set_ylim(.45,4.65)
    ax.set_xticks([]);ax.set_yticks([])
    for sp in ax.spines.values():sp.set_visible(False)
    if not n:ax.text(0,2.6,'NO LOCATION DATA',fontsize=8,color=MUTED,ha='center',va='center')
    elif n<MIN_DENSITY_N:ax.text(-1.90,4.24,'LOW N · POINTS',fontsize=6.5,color=MUTED)
    return render_fig(fig)

In [8]:
def header(c,p,pageno,total=2,df=None):
    shape(c,0,0,14,1.23,MAROON)
    txt(c,.46,.36,'FALMOUTH COMMODORES',11,WHITE,True)
    fit_txt(c,7.0,.14,p.get('name','Unknown'),7.7,25,17,WHITE,True,'center')

    bats = str(p.get('bats','')).upper()
    handedness = {'L':'LEFT HANDED HITTER', 'R':'RIGHT HANDED HITTER', 'S':'SWITCH HITTER'}.get(bats,'')
    if handedness:
        fit_txt(c,7.0,.765,handedness,8.0,12.5,10.5,WHITE,True,'center')
    txt(c,13.56,.37,'HITTER ADVANCE SCOUTING',9,WHITE,True,'right')
    txt(c,13.56,.78,f'PAGE {pageno} OF {total}',7.8,'#F0D4DC',align='right')
    if DEMO_MODE:
        shape(c,0,1.23,14,.17,'#F8E9C9')
        txt(c,7,1.24,'SYNTHETIC DEMONSTRATION  •  NOT A REAL PLAYER / REAL REPORT',7.1,MAROON,True,'center')

def footer(c,pageno,audit):
    shape(c,.35,7.72,13.3,.01,LINE)
    txt(c,13.63,7.76,f'{pageno} / 2',6.55,MUTED,align='right')

def kpi(c,x,y,w,h,label,val):
    shape(c,x,y,w,h,WHITE,LINE)
    txt(c,x+w/2,y+.10,label,7.1,MUTED,True,'center')
    fit_txt(c,x+w/2,y+.31,val,w-.09,14.3,10.5,INK,True,'center')

def draw_overview(c,p,df,audit):
    header(c,p,1,df=df)
    contact=contact_metrics(df,p.get('bats','L'))
    rates=basic_rates(df)
    season=p.get('season',{}) or {}
    calculated={'PITCHES':str(len(df)),'BIP':str(contact['bip']),'MAX EV':fmt(contact['max_ev']),
                'Z-CONTACT':fmt(rates['z_contact'],'pct'),
                'AVG EV':fmt(contact['ev']),'HH%':fmt(contact['hh'],'pct'),
                'CHASE%':fmt(rates['chase'],'pct'),'SWING%':fmt(rates['swing'],'pct'),
                'WHIFF%':fmt(rates['whiff'],'pct'),'Z-SWING%':fmt(rates['z_swing'],'pct')}
    if len(KPI_FIELDS)!=10:raise ValueError('KPI_FIELDS must contain exactly 10 labels for this layout.')
    stats=[(label,calculated.get(label,season.get(label,'—'))) for label in KPI_FIELDS]
    for i,(label,val) in enumerate(stats): kpi(c,.35+i*1.343,1.52,1.245,.64,label,str(val))
    card(c,.35,2.28,5.12,3.19,'SPRAY CHART')
    image_in(c,plot_spray(df,p.get('bats','L')),.47,2.65,4.85,2.67)
    card(c,5.60,2.28,4.55,3.19,'CONTACT QUALITY')
    image_in(c,plot_ev_la(df),5.77,2.69,4.26,2.67)
    card(c,10.28,2.28,3.37,3.19,'BATTED BALL PROFILE','% of classified BIP')
    xx=10.52;ww=2.85
    for i,lab in enumerate(['GB','LD','FB','PU']):
        draw_bar(c,xx,2.77+i*.49,ww,lab,contact.get(lab.lower()),GREEN)
    shape(c,10.45,4.82,3.0,.012,LINE)
    txt(c,10.52,4.93,'DIRECTION',7.3,INK,True)
    parts=[('Pull',contact['pull']),('Middle',contact['middle']),('Oppo',contact['oppo'])]
    for i,(lab,val) in enumerate(parts):
        txt(c,10.58+i*.94,5.12,f'{lab} {fmt(val,"pct",0)}',7.0,BLACK,align='left')
    card(c,.35,5.57,8.01,2.05,'CONTACT BY PITCH FAMILY',None)
    cols=[('PITCH FAMILY',1.35),('BIP',.49),('AVG EV',.65),('MAX EV',.71),('HH%',.66),('GB%',.67),('LD%',.67),('FB%',.67),('PULL%',.73),('OPPO%',.73)]
    widths=np.array([v for _,v in cols]);widths=widths/widths.sum()*7.69
    x0=.49;y0=6.00
    shape(c,x0,y0,7.69,.30,LIGHT)
    xpos=x0
    for (label,_),width in zip(cols,widths):txt(c,xpos+width/2,y0+.075,label,6.25,MUTED,True,'center');xpos+=width
    types=family_rows(df,p.get('bats','L'))
    for i,row in enumerate(types):
        yy=y0+.32+i*.263
        if i%2==1:shape(c,x0,yy,7.69,.255,'#F8FAFB')
        vals=[row['family'],str(row['bip']),fmt(row['ev']),fmt(row['max_ev']),fmt(row['hh'],'pct',0),
              fmt(row['gb'],'pct',0),fmt(row['ld'],'pct',0),fmt(row['fb'],'pct',0),fmt(row['pull'],'pct',0),fmt(row['oppo'],'pct',0)]
        pos=x0
        for j,(v,width) in enumerate(zip(vals,widths)):
            fit_txt(c,pos+width/2,yy+.09,v,width-.03,7.35 if j==0 else 7.0,6.0,INK,j==0,'center')
            pos+=width

    card(c,8.48,5.57,5.17,2.05,'PITCH FAMILY TENDENCIES',None)
    x=8.61; y=5.95
    cols=[('FAMILY',1.13),('N',.43),('SWING',.66),('WHIFF',.66),('CHASE',.66),('AVG EV',.74),('HH',.57)]
    widths=np.array([cw for _,cw in cols]);widths=widths/widths.sum()*4.89
    shape(c,x,y,4.89,.28,LIGHT)
    pos=x
    for (head,_),cw in zip(cols,widths):
        txt(c,pos+cw/2,y+.078,head,6.5,MUTED,True,'center');pos+=cw
    for i,(fam,name) in enumerate([('FB','FASTBALLS'),('BB','BREAKING'),('OS','OFFSPEED')]):
        d=df[df['Family']==fam]
        rate=basic_rates(d); hit=contact_metrics(d,p.get('bats',''))
        yy=y+.30+i*.24
        if i%2:shape(c,x,yy,4.89,.23,'#F6F8FA')
        data=[name,str(len(d)),fmt(rate['swing'],'pct',0),fmt(rate['whiff'],'pct',0),
              fmt(rate['chase'],'pct',0),fmt(hit['ev']),fmt(hit['hh'],'pct',0)]
        pos=x
        for j,(val,cw) in enumerate(zip(data,widths)):
            fit_txt(c,pos+cw/2,yy+.075,val,cw-.01,7.15 if j==0 else 6.95,5.8,INK,j==0,'center')
            pos+=cw
    shape(c,x,7.02,4.89,.012,LINE)

    two=df[(df['Strikes']==2)]
    tr=basic_rates(two)
    fit_txt(c,x,7.11,'TWO STRIKE APPROACH',2.0,7.2,6.6,MAROON,True)
    fit_txt(c,x+2.15,7.11,f'{len(two)} pitches  ·  Swing {fmt(tr["swing"],"pct",0)}  ·  Whiff {fmt(tr["whiff"],"pct",0)}',
            2.74,7.2,6.5,INK)

    footer(c,1,audit)
    c.showPage()

def draw_count_table(c,x,y,w,p,df,hand):
    card(c,x,y,w,1.76,f'VS {hand}','APPROACH')
    out=count_rows(df,hand,(p.get('official_splits',{}) or {}).get(hand,{}))
    colnames=['SPLIT','N','BIP','SWING','Z-SWING','WHIFF','CHASE','Z-CONTACT']
    widths=np.array([1.00,.80,.56,.56,.75,.96,.76,.79]);widths/=widths.sum();widths*=w-.27
    x0=x+.14;y0=y+.45
    shape(c,x0,y0,w-.27,.265,LIGHT)
    pos=x0
    for lab,cw in zip(colnames,widths):txt(c,pos+cw/2,y0+.075,lab,6.4,MUTED,True,'center');pos+=cw
    for i,row in enumerate(out):
        yy=y0+.28+i*.17
        if i%2==1:shape(c,x0,yy,w-.27,.17,'#F6F8FA')
        vals=[row['group'],str(row['N']),str(row['bip']),fmt(row['swing'],'pct'),
              fmt(row['z_swing'],'pct'),fmt(row['whiff'],'pct'),fmt(row['chase'],'pct'),fmt(row['z_contact'],'pct')]
        pos=x0
        for j,(v,cw) in enumerate(zip(vals,widths)):
            fit_txt(c,pos+cw/2,yy+.028,v,cw-.02,7.0,6.0,INK,j==0,'center');pos+=cw

def map_card(c,x,y,w,h,df,hand,family,kind):
    is_whiff=kind=='WHIFF'
    sub=df[df['Hand']==hand].copy()
    if family=='NON_FB':
        sub=sub[sub['Family'].isin(['BB','OS'])]
    else:
        sub=sub[sub['Family']==family]
    selected=sub[sub['Whiff']] if is_whiff else sub[sub['HardHit']]
    valid=selected[selected['PlateLocHeight'].notna()&selected['PlateLocSide'].notna()]
    label=('FASTBALL' if family=='FB' else 'NON FASTBALL')+' / '+('WHIFF' if is_whiff else 'HARD HIT')
    shape(c,x,y,w,h,WHITE,LINE)
    shape(c,x,y,.037,h,BLUE if is_whiff else MAROON)
    fit_txt(c,x+.12,y+.10,label,w-1.0,8.1,6.2,INK,True)
    image_in(c,heatmap_fig(valid,is_whiff),x+.09,y+.37,w-.18,h-.41)

def draw_splits(c,p,df,audit):
    header(c,p,2,df=df)
    draw_count_table(c,.35,1.53,6.55,p,df,'RHP')
    draw_count_table(c,7.04,1.53,6.61,p,df,'LHP')
    for i,(hand,yy) in enumerate([('RHP',3.36),('LHP',5.50)]):
        shape(c,.35,yy,13.30,.27,BLACK)
        txt(c,.51,yy+.045,'VS '+hand+'  •  FASTBALL / NON FASTBALL LOCATION RESULTS',8,WHITE,True)
        txt(c,13.50,yy+.05,"PITCHER'S POV",6.5,WHITE,align='right')
        y=yy+.31
        widths=[3.19]*4; xs=[.35,3.74,7.13,10.52]
        for j,(fam,kind) in enumerate([('FB','HARD HIT'),('FB','WHIFF'),('NON_FB','HARD HIT'),('NON_FB','WHIFF')]):
            map_card(c,xs[j],y,widths[j],1.73,df,hand,fam,kind)
    footer(c,2,audit)
    c.showPage()

def make_pptx(pdf_path,pptx_path,previews_dir):
    try:
        from pptx import Presentation
        from pptx.util import Inches
        prs=Presentation();prs.slide_width=Inches(14);prs.slide_height=Inches(8)
        layout=prs.slide_layouts[6]
        for num in range(2):
            page=fitz.open(pdf_path)
            pix=page[num].get_pixmap(matrix=fitz.Matrix(1.75,1.75),alpha=False)
            png=previews_dir/f'{Path(pdf_path).stem}_page_{num+1}.png'
            pix.save(str(png))
            slide=prs.slides.add_slide(layout)
            slide.shapes.add_picture(str(png),0,0,width=prs.slide_width,height=prs.slide_height)
            page.close()
        prs.save(pptx_path)
        return True
    except ImportError:
        warnings.warn('python-pptx not installed; PDF still generated; skipping PPTX.');return False

def write_report(profile,df,audit,root=OUTPUT_DIR):
    root=Path(root);pdf_dir=root/'pdf';ppt_dir=root/'pptx';prev=root/'previews';meta_dir=root/'audit'
    for directory in (pdf_dir,ppt_dir,prev,meta_dir):directory.mkdir(parents=True,exist_ok=True)
    key=normalize_name(profile['name'])
    if profile.get('batter_id') and 'BatterId' in df.columns:
        player_id = pd.to_numeric(profile['batter_id'], errors='coerce')
        selected = df.loc[
            pd.to_numeric(df['BatterId'], errors='coerce').eq(player_id)
        ].copy()
    else:
        selected = df.loc[
            df['Batter'].map(normalize_name).eq(key)
        ].copy()
    if selected.empty:
        warnings.warn('NO rows for '+profile['name']+'; skipping')
        return None
    target=safe_name(profile['name'])
    pdf_path=pdf_dir/(target+'_Hitter_Advance_Report.pdf')
    c=canvas.Canvas(str(pdf_path),pagesize=(W,H),pageCompression=1)
    c.setTitle(f'Falmouth Hitter Advance Scouting | {profile["name"]}'+(' [DEMO]' if DEMO_MODE else ''))
    local_audit={**audit,'player':profile['name'],'matched_rows':len(selected),'demo_mode':DEMO_MODE,
                 'overall_rates':basic_rates(selected),'contact':contact_metrics(selected,profile.get('bats','')),
                 'by_family':family_rows(selected,profile.get('bats','')),
                 'by_pitch_type':pitch_type_rows(selected,profile.get('bats',''),max_types=100),
                 'split_rhp':count_rows(selected,'RHP',(profile.get('official_splits',{}) or {}).get('RHP',{})),
                 'split_lhp':count_rows(selected,'LHP',(profile.get('official_splits',{}) or {}).get('LHP',{})),
                 'metric_definitions':{'HH%':f'EV>={HARD_HIT_MPH} / EV-known BIP',
                                       'Whiff%':'Swinging misses / all swings',
                                       'Chase%':'Swings out of fixed approx zone / tracked out-of-zone pitches',
                                       'BA SLG':'Not computed: incomplete PA and hit coverage',
                                       'FB':'Fastballs: four-seam, sinker, cutter, two-seam variants',
                                       'BB':'Breaking balls: slider, sweeper, curve, slurve, knuckle-curve',
                                       'OS':'Offspeed: changeup, splitter, forkball, knuckleball'}}
    draw_overview(c,profile,selected,local_audit)
    draw_splits(c,profile,selected,local_audit)
    c.save()
    (meta_dir/(target+'_audit.json')).write_text(json.dumps(local_audit,indent=2,default=str),encoding='utf-8')
    pptx_path=ppt_dir/(target+'_Hitter_Advance_Report.pptx')
    has_pptx=make_pptx(pdf_path,pptx_path,prev)
    return {'name':profile['name'],'pdf':pdf_path,'pptx':pptx_path if has_pptx else None,
            'preview':prev/(pdf_path.stem+'_page_1.png'),'audit':meta_dir/(target+'_audit.json')}

In [9]:
REPORT_FILES=[]
for hitter in HITTERS:
    result=write_report(hitter,DATA,INPUT_AUDIT)
    if result:
        REPORT_FILES.append(result)
        print('\n',result['name'])
        for name,path in result.items():
            if name!='name' and path:print(' ',name,':',path.resolve())
print('\nReports generated:',len(REPORT_FILES),'of',len(HITTERS))


 Charlie Bates
  pdf : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\pdf\Charlie_Bates_Hitter_Advance_Report.pdf
  pptx : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\pptx\Charlie_Bates_Hitter_Advance_Report.pptx
  preview : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\previews\Charlie_Bates_Hitter_Advance_Report_page_1.png
  audit : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\audit\Charlie_Bates_audit.json

 Levi Clark
  pdf : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\pdf\Levi_Clark_Hitter_Advance_Report.pdf
  pptx : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\pptx\Levi_Clark_Hitter_Advance_Report.pptx
  preview : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hitter_Advance_Hitters\previews\Levi_Clark_Hitter_Advance_Report_page_1.png
  audit : C:\Users\brend\OneDrive\School OneDrive\Jupyter\FAL_Hi